# W15-D5 配套实验：Backlog 依赖排序的合法性证明 × 主仓同步机制的量化依据

与 md 的分工：md 给出**结论**（四批 backlog + S1-S6 机制），本 notebook 回答两个「凭什么」：
1. **排序凭什么合法**——把 11 项工作 + 12 条依赖边建成 DAG，Kahn 拓扑排序验证 md 批次划分无违边，最长路给出最少串行批次数；
2. **机制凭什么这样定**——用 lnkcre 主仓 2026-08-28→09-10 的**真实日提交分布**（14 天 211 commits，均值 15.1/天）驱动四种同步策略模拟，量化感知延迟与深读成本；再用 4-8 月真实表增长（68/152/148/98）模拟 digest 周期对孤儿表滞留的影响。

全部为 CPU 小规模模拟，无网络、无重型依赖。


In [ ]:
# -*- coding: utf-8 -*-
# W15-D5 标准字体配置（TOOLS.md 方式）
from matplotlib import font_manager
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from collections import deque

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("字体:", font_name)

W15 = "/root/learning-notebooks/第15周"


## §1 Backlog DAG：拓扑合法性 × 最少批次数

节点 = md §2 的 11 项工作；边 = md 表格中「消费的构件」列声明的依赖（每条边带理由）。
验证三件事：① 12 条边在 Kahn 拓扑序中全部前向（DAG 无环）；② md 的批次划分（W16/W17/W18/W19）对每条边满足 `批次[前驱] < 批次[后继]`（同批禁止有依赖）；③ 最长路径长度（节点数）= 最少串行批次数，对照 md 的 4 批。


In [ ]:
# 节点：md §2 backlog 表的 11 项工作
ITEMS = {
    "G01":     "G-01 治理头走 openspec change",
    "G05":     "G-05 registry 锚点 + frozen CI",
    "CHATBI":  "chatbi 白名单 20 对象登记 Identity",
    "G07":     "G-07 新表挡板（citing 模式）",
    "G04":     "G-04 术语消歧第二批（白名单域）",
    "MCP":     "MCP 工具描述生成 change",
    "RULE":    "lease→cash 链 Rule 显式化",
    "POLICY":  "约束声明 draft→0.1 评审",
    "G06":     "G-06 双迁移分叉裁决",
    "VERSION": "策略载体版本化统一裁决",
    "BREAKFIX":"三断链修复提案",
}
# 依赖边（u → v：v 消费 u 的产出），理由见 md §2 表格「消费的构件」列
EDGES = [
    ("G01", "G07",      "挡板登记进受治理 SoT（无治理头的挡板=第二套无主清单）"),
    ("G01", "G04",      "消歧结论落 ontology（直改无治理头文件违 S5 归档律）"),
    ("G01", "MCP",      "工具描述的术语引用落受治理 SoT"),
    ("CHATBI", "G04",   "先知白名单 20 对象，才定先消歧哪些域内术语"),
    ("CHATBI", "G07",   "白名单 citing action 就是挡板的实现范本"),
    ("CHATBI", "MCP",   "消费链管线（生成→导入→回执）先跑通再复制到第二消费方"),
    ("G05", "RULE",     "frozen 无 CI 时显式化规则=往漏桶加水"),
    ("G05", "POLICY",   "声明的 evidence 载体锚定依赖 registry 锚点"),
    ("G05", "G06",      "表归属锚定后，才量得出双迁移分叉有多大"),
    ("POLICY", "VERSION","version_binding 字段定稿，才知道要版本化什么"),
    ("POLICY", "BREAKFIX","断链提案引用约束声明的证据格式"),
    ("G07", "RULE",     "规则显式化的表域先有挡板，防显式化中途新表涌入"),
]
# md §2 的批次划分
BATCH = {"G01": 16, "G05": 16, "CHATBI": 16,
         "G07": 17, "G04": 17, "MCP": 17,
         "RULE": 18, "POLICY": 18, "G06": 18,
         "VERSION": 19, "BREAKFIX": 19}

# --- Kahn 拓扑排序 ---
adj = {u: [] for u in ITEMS}
indeg = {u: 0 for u in ITEMS}
for u, v, _ in EDGES:
    adj[u].append(v); indeg[v] += 1
q = deque(sorted([u for u in ITEMS if indeg[u] == 0]))
topo = []
while q:
    u = q.popleft(); topo.append(u)
    for v in sorted(adj[u]):
        indeg[v] -= 1
        if indeg[v] == 0: q.append(v)
assert len(topo) == len(ITEMS), "DAG 有环！"
pos = {u: i for i, u in enumerate(topo)}
viol_topo = [(u, v) for u, v, _ in EDGES if pos[u] > pos[v]]
viol_batch = [(u, v) for u, v, _ in EDGES if not (BATCH[u] < BATCH[v])]
print(f"① Kahn 拓扑序（{len(topo)} 节点，DAG 无环）:", " → ".join(topo))
print(f"② 拓扑序违边: {viol_topo if viol_topo else '无（12 条边全部前向）'}")
print(f"③ md 批次违边: {viol_batch if viol_batch else '无（W16<W17<W18<W19 划分合法）'}")

# --- 最长路径（节点数）= 最少串行批次 ---
level = {u: 1 for u in ITEMS}
for u in topo:
    for v in adj[u]:
        level[v] = max(level[v], level[u] + 1)
L = max(level.values())
# 回溯一条最长链
chain = []
cur = max(ITEMS, key=lambda u: (level[u], u))
chain.append(cur)
while level[cur] > 1:
    cur = [u for u in ITEMS if cur in adj[u] and level[u] == level[cur] - 1][0]
    chain.append(cur)
chain = chain[::-1]
print(f"④ 最长依赖链 = {L} 节点 → 最少串行批次 = {L}；示例链:", " → ".join(chain))
print(f"⑤ md 划 {max(BATCH.values()) - min(BATCH.values()) + 1} 批（W16-W19），比最少批数多 {4 - L} 批 = 结构性缓冲一周")
crit_start = [u for u in ITEMS if level[u] == 1]
print(f"⑥ 关键链起点（W16 批次）:", ", ".join(sorted(crit_start)), "—— 治理地基与消费面事实化卡着 v0.2 主菜进度")


In [ ]:
# --- DAG 可视化（按层级布局，批次着色）---
fig, ax = plt.subplots(figsize=(12, 7.5))
lv_nodes = {}
for u in ITEMS: lv_nodes.setdefault(level[u], []).append(u)
xy = {}
for lv, nodes in lv_nodes.items():
    for i, u in enumerate(sorted(nodes)):
        xy[u] = (lv, (i - (len(nodes) - 1) / 2) * 1.15)
batch_color = {16: "#2e7d32", 17: "#1565c0", 18: "#e65100", 19: "#6a1b9a"}
batch_name = {16: "W16 治理地基+消费面", 17: "W17 消费链扩展+挡板", 18: "W18 规则显式化", 19: "W19 收口"}
for u, v, reason in EDGES:
    x1, y1 = xy[u]; x2, y2 = xy[v]
    ax.annotate("", xy=(x2 - 0.12, y2), xytext=(x1 + 0.12, y1),
                arrowprops=dict(arrowstyle="->", color="#888", lw=1.2,
                                connectionstyle="arc3,rad=0.12"))
for u in ITEMS:
    x, y = xy[u]
    ax.scatter([x], [y], s=3800, c=batch_color[BATCH[u]], alpha=0.92, zorder=3,
               edgecolors="white", linewidths=2)
    ax.text(x, y + 0.02, u, ha="center", va="center", color="white",
            fontsize=10.5, fontweight="bold", zorder=4)
    ax.text(x, y - 0.34, ITEMS[u][:16], ha="center", va="top", fontsize=6.8, color="#333", zorder=4)
handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=batch_color[b],
           markersize=11, label=batch_name[b]) for b in sorted(batch_name)]
ax.legend(handles=handles, loc="lower right", fontsize=9, framealpha=0.95)
ax.set_xlim(0.3, L + 0.9); ax.set_ylim(-1.9, 1.9)
ax.set_xticks(range(1, L + 1))
ax.set_xticklabels([f"层级{lv}\n(第{lv}串行批)" for lv in range(1, L + 1)], fontsize=9)
ax.set_yticks([])
ax.set_title("W16+ Backlog 依赖 DAG（11 项工作 / 12 条依赖边 / md 四批划分拓扑合法）", fontsize=13)
for s in ["top", "right"]: ax.spines[s].set_visible(False)
plt.tight_layout()
p1 = f"{W15}/w15d5_backlog_dag.png"
plt.savefig(p1, dpi=130); plt.close()
print("已保存:", p1)


## §2 同步策略模拟：真实日提交分布驱动的四种策略对比

到达过程：lnkcre 主仓 8/28-9/10 **实测日提交** `[12,27,17,17,15,13,2,7,11,26,19,13,18,14]` 作 14 天模板循环 12 周（84 天），Poisson 抖动保持突发性；第 5-6 周叠加 ×3 的「双 wave 齐发」压力场景（模拟 R-wave + analysis wave 同期落地，即历史上 714 积累的速率形态）。

策略：**A** 日深扫（学习期形态）｜**B** 纯周扫｜**C** 周扫+日探针·阈值100（S1+S2+S3）｜**D** = C+挡板300（S4，同检测逻辑，多计紧急对齐）｜**E** 无雷达（714 场景重演）。
指标：感知延迟（commit 落地→被深 digest 看到）、深读次数（打断开发流的成本）、最大未感知积压。


In [ ]:
# --- 到达过程 ---
rng = np.random.default_rng(20260911)
REAL_DAYS = [12, 27, 17, 17, 15, 13, 2, 7, 11, 26, 19, 13, 18, 14]  # 8/28-9/10 实测
NDAYS = 84  # 12 周
arrivals = np.array([rng.poisson(REAL_DAYS[d % 14]) for d in range(NDAYS)], dtype=float)
burst = np.zeros(NDAYS); burst[35:49] = 2.0  # 第 6-7 周(0起算35-48)速率 ×3
arrivals = arrivals * (1 + burst)
print(f"总 commits: {arrivals.sum():.0f}（常速段均值 {arrivals[:35].mean():.1f}/天，压力段均值 {arrivals[35:49].mean():.1f}/天）")

def simulate(arr, mode):
    """返回每个 commit 的感知延迟(天)、深读次数、探针次数、逐日未感知积压"""
    lags, deep_days, probe_days, pending_track = [], [], [], []
    pending_commits = []  # (day, ) 待感知
    deep_cnt = probe_cnt = emerg_cnt = 0
    for d in range(len(arr)):
        n = arr[d]
        for _ in range(int(n)): pending_commits.append(d)
        # 日探针（B/A/E 无探针语义；C/D 每天秒级计数）
        do_deep = False
        if mode == "A":
            do_deep = True
        elif mode == "B":
            do_deep = (d % 7 == 0)
        elif mode in ("C", "D"):
            probe_cnt += 1
            if len(pending_commits) > 100 or d % 7 == 0:
                do_deep = True
                if len(pending_commits) > 300 and mode == "D": emerg_cnt += 1
        if do_deep:
            deep_cnt += 1; deep_days.append(d)
            lags.extend(d - np.array(pending_commits))
            pending_commits = []
        pending_track.append(len(pending_commits))
    # 期末未感知的 commits 记为「至今未感知」
    lags_arr = np.array(lags) if lags else np.array([0.0])
    return dict(lags=lags_arr, deep=deep_cnt, probe=probe_cnt, emerg=emerg_cnt,
                pending=np.array(pending_track), leftover=len(pending_commits))

results = {m: simulate(arrivals, m) for m in ["A", "B", "C", "D", "E"]}
print(f"{'策略':<4}{'深读':>5}{'探针':>5}{'紧急对齐':>6}{'均滞后':>8}{'P95滞后':>8}{'最大滞后':>8}{'最大未感知积压':>10}{'期末滞留':>8}")
for m, r in results.items():
    lag = r["lags"]
    print(f"{m:<4}{r['deep']:>5}{r['probe']:>5}{r['emerg']:>6}"
          f"{lag.mean():>8.2f}{np.percentile(lag,95):>8.1f}{lag.max():>8.0f}"
          f"{r['pending'].max():>10.0f}{r['leftover']:>8}")
print()
c, b, e = results["C"], results["B"], results["E"]
print(f"结论1: C 相比 A 深读成本降至 {c['deep']/results['A']['deep']*100:.0f}%（{c['deep']} vs {results['A']['deep']} 次）；平均滞后 {c['lags'].mean():.1f} vs {b['lags'].mean():.1f} 天，最大滞后仍 {c['lags'].max():.0f} 天（常规感知由周节奏决定）")
print(f"结论2: C 相比 B 把最大未感知积压从 {b['pending'].max():.0f} 压到 {c['pending'].max():.0f} commits（探针日日在线，>100 当天转深读）")
print(f"结论3: E（无雷达）期末滞留 {e['leftover']} commits 未感知——714 场景的机制重演；压力段探针在 100 线先熔断转深读，挡板 300 全程未触及（紧急对齐 {results['D']['emerg']} 次）——保险丝分层的预期行为：100 先救感知，300 只在探针失效时救开发")


In [ ]:
# --- 策略对比图 ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
names = ["A 日深扫\n(学习期)", "B 纯周扫", "C 周扫+探针\n(S1-S3)", "D C+挡板300\n(+S4)", "E 无雷达"]
keys = list(results.keys())
colors = ["#90a4ae", "#64b5f6", "#2e7d32", "#1565c0", "#c62828"]
mean_lag = [results[k]["lags"].mean() for k in keys]
max_lag = [results[k]["lags"].max() for k in keys]
x = np.arange(len(keys))
axes[0].bar(x - 0.18, mean_lag, 0.36, label="平均滞后(天)", color="#42a5f5")
axes[0].bar(x + 0.18, max_lag, 0.36, label="最大滞后(天)", color="#0d47a1")
axes[0].set_xticks(x); axes[0].set_xticklabels(names, fontsize=8)
axes[0].set_title("感知延迟（commit 落地→被深读看到）", fontsize=11); axes[0].legend(fontsize=8)
deep = [results[k]["deep"] for k in keys]
axes[1].bar(x, deep, 0.55, color=colors)
axes[1].set_xticks(x); axes[1].set_xticklabels(names, fontsize=8)
axes[1].set_title("深 digest 次数 / 84 天（打断开发流的成本）", fontsize=11)
for i, v in enumerate(deep): axes[1].text(i, v + 1, str(v), ha="center", fontsize=9)
axes[2].plot(range(NDAYS), results["E"]["pending"], color="#c62828", lw=1.6, label="E 无雷达")
axes[2].plot(range(NDAYS), results["B"]["pending"], color="#64b5f6", lw=1.4, label="B 纯周扫")
axes[2].plot(range(NDAYS), results["C"]["pending"], color="#2e7d32", lw=1.4, label="C 周扫+日探针")
axes[2].axhline(100, color="#f9a825", ls="--", lw=1, label="S3 触发线 100")
axes[2].axhline(300, color="#b71c1c", ls=":", lw=1.2, label="S4 挡板 300")
axes[2].set_title("未感知积压曲线（第 6-7 周 ×3 压力场景）", fontsize=11)
axes[2].set_xlabel("天"); axes[2].set_ylabel("behind (commits)"); axes[2].legend(fontsize=7.5)
for a in axes: a.spines["top"].set_visible(False); a.spines["right"].set_visible(False)
plt.suptitle("同步策略对比（到达过程 = 主仓真实日分布 + 压力场景）", fontsize=13)
plt.tight_layout()
p2 = f"{W15}/w15d5_sync_strategies.png"
plt.savefig(p2, dpi=130); plt.close()
print("已保存:", p2)


## §3 孤儿表稀释模拟：digest 周期 × 表增长速率

v0.1 熵增基线：月度新表 **68 → 152 → 148 → 98**（4/6/7/8 月），跨切面桶 26.2%。新 canonical 表在「无 Context 归属登记」前是孤儿——登记只发生在深 digest 时（孤儿收敛靠事后启发式 = G-07 的现状）。模拟 4 个月（120 天）分时段到达，比较 digest 周期 1/7/14/30 天与从不 digest 的孤儿滞留。


In [ ]:
# --- 孤儿表稀释模拟 ---
rng2 = np.random.default_rng(42)
MONTHLY = [68, 152, 148, 98]  # 4/6/7/8 月实测新表
ND = 120
orphan_arrival = np.concatenate([rng2.poisson(m / 30, 30) for m in MONTHLY]).astype(float)
print(f"4 个月模拟新表总数: {orphan_arrival.sum():.0f}（对照实测合计 {sum(MONTHLY)}）")

def orphan_sim(arr, cadence):
    pend, ages, curve = [], [], []
    for d in range(len(arr)):
        pend.extend([d] * int(arr[d]))
        if cadence is not None and (d + 1) % cadence == 0:
            ages.extend([(d + 1) - c for c in pend]); pend = []
        curve.append(len(pend))
    return np.array(curve), np.array(ages) if ages else np.array([np.nan]), len(pend)

cads = [(1, "日 digest"), (7, "周 digest (S1)"), (14, "双周"), (30, "月 digest"), (None, "从不 (G-07 现状)")]
print(f"{'周期':<16}{'平均滞留(天)':>12}{'P95滞留':>8}{'峰值孤儿数':>10}{'期末孤儿':>8}")
sim_res = {}
for cad, name in cads:
    curve, ages, left = orphan_sim(orphan_arrival, cad)
    sim_res[name] = (curve, ages, left)
    print(f"{name:<16}{np.nanmean(ages):>12.1f}{np.nanpercentile(ages,95):>8.0f}{curve.max():>10.0f}{left:>8}")
print()
print("结论: 周把平均孤儿滞留压到 ~4 天、峰值 ~34（由月度到达高峰决定）；月 digest 峰值 ~149 破百；")
print("     从不登记 → 全部滞留（G-07 现状）——语义资产不是被推翻死的，是被稀释死的（v0.1 熵增基线）")


In [ ]:
# --- 稀释图 ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
show = ["日 digest", "周 digest (S1)", "双周", "月 digest", "从不 (G-07 现状)"]
cols = ["#9ccc65", "#2e7d32", "#f9a825", "#e65100", "#c62828"]
for name, c in zip(show, cols):
    curve, _, _ = sim_res[name]
    axes[0].plot(range(ND), curve, lw=1.5, color=c, label=name)
axes[0].set_title("孤儿表存量曲线（月度到达率 68/152/148/98）", fontsize=11)
axes[0].set_xlabel("天"); axes[0].set_ylabel("未登记孤儿表数"); axes[0].legend(fontsize=8)
names2 = [n for n in show]
mean_age = [np.nanmean(sim_res[n][1]) for n in names2]
axes[1].bar(range(len(names2)), mean_age, 0.55, color=cols)
axes[1].set_xticks(range(len(names2))); axes[1].set_xticklabels(names2, fontsize=8, rotation=12)
axes[1].set_title("孤儿平均滞留天数（新表诞生→被语义资产登记）", fontsize=11)
for i, v in enumerate(mean_age):
    axes[1].text(i, v + (2 if np.isfinite(v) else 0), f"{v:.1f}" if np.isfinite(v) else "∞", ha="center", fontsize=9)
axes[1].set_ylim(0, max([a for a in mean_age if np.isfinite(a)]) * 1.18)
for a in axes: a.spines["top"].set_visible(False); a.spines["right"].set_visible(False)
plt.suptitle("digest 周期对语义资产稀释度的影响（G-07 挡板的量化依据）", fontsize=13)
plt.tight_layout()
p3 = f"{W15}/w15d5_orphan_dilution.png"
plt.savefig(p3, dpi=130); plt.close()
print("已保存:", p3)


## 结论（回填 md 的三个「凭什么」）

1. **排序合法性**：11 节点 12 边 DAG 无环，md 四批划分零违边；最长链 3 节点 → 最少 3 个串行批，md 划 4 批留一周缓冲；关键链起点（G-01 / G-05 / CHATBI）全在 W16——**治理地基和消费面事实化是 v0.2 主菜的前置，不是可选项**。
2. **同步机制分层依据**：真实日分布（均值 15.1、突发 27）+ 压力场景下，S1-S3（周深读+日探针+阈值触发）以 ~1/4 的深读成本（20 vs 84 次）把最大未感知积压从 309 压到 99（探针阈值附近）；常规感知延迟均值 2.2 天、最大 6 天（由周节奏决定），危险积压（>100）的暴露时间 ≤1 天；无雷达策略期末滞留全部 commits——714 场景的机制重演。挡板 300 在当前速率下是保险丝（不常态触发），价值在速率突变时强制「先对齐再开发」。
3. **稀释量化**：月增 ~100-150 表的速率下，周 digest 的孤儿平均滞留 ~4 天、峰值 ~34；月 digest 峰值 ~149 破百；从不登记 = 全部滞留——**G-07 新表挡板（W17）的紧迫性来自这条曲线，也解释了 v0.1 熵增基线为什么把「第一个消费方要赶在稀释前面」写成 P0 语气**。
